## Black-Scholes-Merton Option pricing model

In [1]:
import numpy as np
from scipy.stats import norm 
import QuantLib as ql  

In [2]:
def black_scholes(S, X, r, T, sigma, option):
    d1 = (np.log(S/X)+(r+0.5*sigma**2)*T)/(sigma*np.sqrt(T)) 
    d2 = d1-sigma*np.sqrt(T) 

    if option == 'call':
        value = S*norm.cdf(d1)-X*np.exp(-r*T)*norm.cdf(d2)
    elif option == 'put':
        value = X*np.exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)
    else:
        raise ValueError('Option is either call or put') 

    return value   

## Binomial Option Pricing Model

In [3]:
def binomial_option_pricing_model(S, X, r, T, sigma, steps, option):
    dt = T/steps
    u = np.exp(sigma*np.sqrt(dt)) 
    d = 1/u
    p = (np.exp(r*dt)-d)/(u-d) 

    prices = np.zeros((steps+1, steps+1)) 
    prices[0, 0] = S

    for i in range(1, steps+1):
        prices[i,0] = prices[i-1, 0]*u
        for j in range(1, i+1):
            prices[i,j] = prices[i-1, j-1]*d 

    option_values = np.zeros((steps+1, steps+1)) 
    
    if option == 'call':
        option_values[:,steps] = np.maximum(0, prices[:,steps]-X)
    elif option == 'put':
        option_values[:,steps] = np.maximum(0, X-prices[:,steps]) 
    else:
        raise ValueError('Option is call or put')

    for i in range(steps-1, -1, -1):
        for j in range(i+1):
            option_values[i,j] = np.exp(-r*dt) * (p * option_values[i + 1, j] + (1 - p) * option_values[i + 1, j + 1])

    return option_values[0,0] 

## Monte Carlo Simulations

In [4]:
def monte_carlo_pricing_model(S, X, r, T, sigma, num_sims, option):
    np.random.seed(42)
    dt = T/252 
    S_t = S*np.exp((r-0.5*sigma**2)*dt + sigma*np.sqrt(dt)*np.random.randn(num_sims))

    if option == 'call':
        payoff = np.maximum(0, S_t-X)
    elif option == 'put':
        payoff = np.maximum(0, X-S_t) 
    else:
        raise ValueError('Option is call or put') 

    option = np.exp(-r*T)*np.mean(payoff) 
     
    return option     

## Heston Model

In [5]:
def heston_option_pricing(S, X, r, T, v0, kappa, theta, sigma, rho, option_type):

    option_type = ql.Option.Call if option_type == 'call' else ql.Option.Put
    exercise_date = ql.Date().todaysDate() + int(T*365)
    payoff = ql.PlainVanillaPayoff(option_type, X) 
    european_exercise = ql.EuropeanExercise(exercise_date)

    spot_handle = ql.QuoteHandle(ql.SimpleQuote(S)) 
    risk_free_curve = ql.YieldTermStructureHandle(ql.FlatForward(0, ql.NullCalendar(), r, ql.Actual365Fixed()))
    dividend_yield = ql.YieldTermStructureHandle(ql.FlatForward(0, ql.NullCalendar(), 0.0, ql.Actual365Fixed()))

    heston_process = ql.HestonProcess(risk_free_curve, dividend_yield, spot_handle, v0, kappa, theta, sigma, rho) 
    
    heston_model = ql.HestonModel(heston_process) 
    engine = ql.AnalyticHestonEngine(heston_model)
    option = ql.VanillaOption(payoff, european_exercise)
    option.setPricingEngine(engine) 

    option_price = option.NPV() 
    return option_price 

In [6]:
S = 100     # Current stock price
X = 100     # Exercise or strike price
r = 0.05    # Risk-free interest rate
T = 1   # Time to expiration (in years)
sigma = 0.2     # Volatility
steps = 100 
num_sims = 10000 

v0 = 0.1  # Initial volatility
kappa = 0.5  # Mean reversion speed
theta = 0.1  # Long-term volatility
sigma = 0.1  # Volatility of volatility
rho = -0.5  # Correlation between asset price and volatility

option = 'call' 

In [7]:
print('Option price using Black-Scoles: ',black_scholes(S, X, r, T, sigma, option))  
print('Option price using binomial model: ',binomial_option_pricing_model(S, X, r, T, sigma, steps, option)) 
print('Option price using Monte Carlo: ',monte_carlo_pricing_model(S, X, r, T, sigma, num_sims, option)) 
print("Option price using Heston Model:", heston_option_pricing(S, X, r, T, v0, kappa, theta, sigma, rho, option))

Option price using Black-Scoles:  6.804957708822144
Option price using binomial model:  0.0
Option price using Monte Carlo:  0.24846143199761683
Option price using Heston Model: 14.813422011039673
